In [2]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import numpy as np
import scienceplots
import tifffile as tiff

from boulder_statistics.analysis.data_product_encyclopedia import DataProductEncyclopedia

plt.style.use('science')
plt.rcParams["figure.figsize"] = (3.5, 3.5 * ((5**0.5 - 1) / 2)) # 3.5
plt.rcParams["figure.dpi"] = 600
%matplotlib inline

import polars as pl
from polars import Expr, LazyFrame, DataFrame
import numpy as np
from pathlib import Path
from typing import Any
import tifffile
from typing import Dict
from typing import Callable


from boulder_statistics.analysis.data_product_encyclopedia import DataProductEncyclopedia

dp = DataProductEncyclopedia(
    data_products_path=Path(r"C:\Users\Joshu\OneDrive - Nexus365\AO33\Boulder_database\AO33\.data_products"))

In [3]:
df = dp.mask_atlas_combined.filter(pl.col("face") == "posx").collect()
df

lod_level,lod_code,face,i,j,row_id,uint8_reflectance,32bit_reflectance,positions_x,positions_y,positions_z
u8,str,str,u32,u32,u32,u8,f32,f32,f32,f32
1,"""A""","""posx""",0,577,19545,150,0.026051,-0.121165,-0.141084,-0.141084
1,"""A""","""posx""",0,589,19545,199,0.03446,-0.120867,-0.141213,-0.141212
1,"""A""","""posx""",0,605,19545,191,0.033173,-0.120426,-0.14134,-0.141346
4,"""AAAB""","""posx""",0,626,61029,129,0.022319,-0.119775,-0.141436,-0.141441
3,"""AAA""","""posx""",0,776,36995,33,0.005656,-0.11541,-0.142441,-0.142436
…,…,…,…,…,…,…,…,…,…,…
1,"""D""","""posx""",8168,7899,2907629,107,0.018463,0.13142,-0.141498,0.140671
1,"""D""","""posx""",8168,7922,2907629,192,0.033255,0.131975,-0.141241,0.14042
1,"""D""","""posx""",8168,8003,2907629,174,0.030146,0.133797,-0.140221,0.139399


In [12]:
def normalize_to_uint8(img: np.ndarray) -> np.ndarray:
    img = img.astype(np.float32)

    if img.ndim == 3 and img.shape[-1] == 4:
        rgb = img[..., :3]
        alpha = img[..., 3].astype(np.uint8)

        min_val = rgb.min()
        max_val = rgb.max()

        if max_val == min_val:
            rgb = np.zeros_like(rgb, dtype=np.uint8)
        else:
            rgb = ((rgb - min_val) * (255.0 / (max_val - min_val))).astype(np.uint8)

        return np.dstack((rgb, alpha))

    min_val = img.min()
    max_val = img.max()

    if max_val == min_val:
        return np.zeros_like(img, dtype=np.uint8)

    return ((img - min_val) * (255.0 / (max_val - min_val))).astype(np.uint8)

In [ ]:
from numpy import where


def render_boulders_for_face(face : str, export_size : int) -> np.ndarray:

    img = np.zeros((8192, 8192, 3), dtype=np.float64)
    full_atlas = dp.Phi_mesh.filter(pl.col("face") == face).collect()
    img[full_atlas["i"].to_numpy(), full_atlas["j"].to_numpy()] = full_atlas["area"].to_numpy()[:, None]

    mean_val = img.mean()

    img = np.divide(
        1.0,
        img / mean_val,
        out=np.ones_like(img, dtype=np.float32),
        where=img != 0,
    )

    img_small = cv2.resize(
        img,
        (export_size, export_size),  # (width, height)
        interpolation=cv2.INTER_AREA,
    )

    return img_small

In [105]:
# import matplotlib.pyplot as plt

# img = render_boulders_for_face("negy", 1024)

# plt.figure(figsize=(8, 4))
# # plt.hist(img.ravel(), bins=np.geomspace(img.min(), img.max(), 250), range=(0, 255))
# plt.hist(img.ravel(), bins=5000, range=(0, 255))
# plt.xlabel("Pixel value")
# plt.ylabel("Count")
# # plt.xscale("log")
# plt.title("Histogram of render_boulders_for_face('posy', 1024)")
# plt.show()

In [106]:
import numpy as np

FACE_SIZE = 1024

faces_LUT = {
    "negy": (1 * FACE_SIZE, 0 * FACE_SIZE),
    "posy": (1 * FACE_SIZE, 2 * FACE_SIZE),
    "negx": (0 * FACE_SIZE, 1 * FACE_SIZE),
    "posz": (1 * FACE_SIZE, 1 * FACE_SIZE),
    "posx": (2 * FACE_SIZE, 1 * FACE_SIZE),
    "negz": (3 * FACE_SIZE, 1 * FACE_SIZE),
}

# RGBA image with transparent background
cubemap = np.zeros((3 * FACE_SIZE, 4 * FACE_SIZE, 4), dtype=np.float32)

for face, (x, y) in faces_LUT.items():
    rgb = render_boulders_for_face(face, FACE_SIZE)  # (1024, 1024, 3)

    cubemap[y:y + FACE_SIZE, x:x + FACE_SIZE, :3] = rgb
    cubemap[y:y + FACE_SIZE, x:x + FACE_SIZE, 3] = 255

cubemap = normalize_to_uint8(cubemap)

In [107]:
cv2.imwrite(".plots/phi_mesh_cubemap.png", cubemap)

True